In [2]:
# ============================================================================
# DATA LOADING 
# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 100)

# Load the optimised features CSV (all 257,673 samples, 7 PCs + target)
dftotal = pd.read_csv('/home/e-ackerson/GIT_hub/Research_Project/DL_models/CNN_models_evals/CNN_IDS-34Features/34_optimal_features_ids.csv')

# Drop the unnamed index column if present
if 'Unnamed: 0' in dftotal.columns:
    dftotal = dftotal.drop(columns=['Unnamed: 0'])

DFtotal = dftotal[[
    'Fused_proto_PC2', 'Fused_sbytes_PC2', 'Fused_proto_PC3',
    'Fused_proto_PC1', 'Fused_proto_PC13', 'Fused_proto_PC7',
    'Fused_sbytes_PC1', 'attack_cat'
]]

print("DFtotal shape:", DFtotal.shape)
print("Unique original classes:", sorted(DFtotal['attack_cat'].unique()))

# ============================================================================
# APPLY MACRO-CLASS MAPPING
# ============================================================================
# Original 10 classes (alphabetical integer encoding):
#   0 Analysis, 1 Backdoor, 2 DoS, 3 Exploits, 4 Fuzzers,
#   5 Generic, 6 Normal, 7 Reconnaissance, 8 Shellcode, 9 Worms

category_map = {
    2: 'DoS-Fuzzers',
    4: 'DoS-Fuzzers',
    1: 'System-Exploits',
    3: 'System-Exploits',
    8: 'System-Exploits',
    9: 'Worm-Recon',
    0: 'Worm-Recon',
    7: 'Worm-Recon',
    5: 'Generic',
    6: 'Normal'
}

DFtotal['attack_cat'] = DFtotal['attack_cat'].map(category_map)

print("\nMacro-class distribution:")
print(DFtotal['attack_cat'].value_counts().sort_index())

# Re-encode macro-classes to contiguous integers (0-4)
from sklearn.preprocessing import LabelEncoder

macro_encoder = LabelEncoder()
DFtotal['attack_cat'] = macro_encoder.fit_transform(DFtotal['attack_cat'])

print("\nMacro-class integer mapping:")
for i, name in enumerate(macro_encoder.classes_):
    print(f"  {i} -> {name}")


print("Macro-class distribution:")
print(DFtotal['attack_cat'].value_counts().sort_index())

# ============================================================================
# TRAIN / TEST SPLIT
# ============================================================================
from sklearn.model_selection import train_test_split

y = DFtotal['attack_cat']
x = DFtotal.drop(columns={'attack_cat'})

xtrain_val, xtest, ytrain_val, ytest = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain:", xtrain_val.shape, "Test:", xtest.shape)


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
FEATURES = [
    'Fused_proto_PC2', 'Fused_sbytes_PC2', 'Fused_proto_PC3',
        'Fused_proto_PC1', 'Fused_proto_PC13', 'Fused_proto_PC7',
        'Fused_sbytes_PC1'
]

x_train = xtrain_val[FEATURES].copy()
x_test  = xtest[FEATURES].copy()

print("x_train:", x_train.shape, "x_test:", x_test.shape)


# ============================================================================
# CLASS WEIGHTS
# ============================================================================
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(ytrain_val)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=ytrain_val
)
class_weights = dict(zip(classes, class_weights_array))
print("Class weights:", class_weights)


# ============================================================================
# RESHAPE FOR 1D-CNN
# ============================================================================
y_train = ytrain_val.values
y_test  = ytest.values
num_classes = len(np.unique(y_train))

n_features = x_train.shape[1]

X_train_cnn = x_train.values.reshape(-1, n_features, 1)
X_test_cnn  = x_test.values.reshape(-1, n_features, 1)

print("X_train_cnn:", X_train_cnn.shape)
print("X_test_cnn: ", X_test_cnn.shape)
print("num_classes:", num_classes)
print("n_features: ", n_features)



# ..............................................................................
#............................... ARTIFACTS ..................................
import os
import joblib
import json

# Create the artifacts directory if it does not exist
os.makedirs('artifacts_designed_34', exist_ok=True)
print("Artifacts directory ready: artifacts_designed_34/")

# ............................................................................
# Saving Macro-class label encoder
# ............................................................................
joblib.dump(macro_encoder, 'artifacts_designed_34/macro_label_encoder.pkl')
print("\nSaved label encoder. Classes:")
for i, name in enumerate(macro_encoder.classes_):
    print(f"  {i} -> {name}")

label_map = {int(i): name for i, name in enumerate(macro_encoder.classes_)}
with open('artifacts_designed_34/label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)
print("Saved label_map.json")

# ............................................................................
# Save feature order
# ............................................................................
FEATURE_ORDER = [
    'Fused_proto_PC2', 'Fused_sbytes_PC2', 'Fused_proto_PC3',
    'Fused_proto_PC1', 'Fused_proto_PC13', 'Fused_proto_PC7',
    'Fused_sbytes_PC1'
]
with open('artifacts_designed_34/feature_order.json', 'w') as f:
    json.dump(FEATURE_ORDER, f, indent=2)
print("\nSaved feature_order.json:", FEATURE_ORDER)

# ............................................................................
# Verify what was saved
# ............................................................................
print("\nArtifacts directory contents:")
for fname in os.listdir('artifacts_designed_34'):
    fpath = os.path.join('artifacts_designed_34', fname)
    size = os.path.getsize(fpath)
    print(f"  {fname}  ({size} bytes)")

# #..............................................................................
# # Save base features for PCA Projection
# #..............................................................................
# BASE_FEATURES = [
#     'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss',
#     'dloss', 'sload', 'dload', 'spkts', 'dpkts', 'swin', 'dwin', 'stcpb',
#     'dtcpb', 'smeansz', 'dmeansz', 'sjit', 'djit', 'sintpkt', 'dintpkt',
#     'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_srv_src',
#     'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm',
#     'ct_dst_sport_ltm', 'ct_dst_src_ltm'
# ]
# with open('artifacts_designed_34/base_features.json', 'w') as f:
#     json.dump(BASE_FEATURES, f, indent=2)
# print(f"Base feature count: {len(BASE_FEATURES)}")




# # ============================================================================
# # CNN MODEL ARCHITECTURE
# # ============================================================================
# import tensorflow as tf
# from tensorflow.keras.layers import (
#     Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D,
#     Dense, Dropout, Concatenate, BatchNormalization
# )
# from tensorflow.keras.models import Model
# from tensorflow.keras.optimizers import Adam

# np.random.seed(42)
# tf.random.set_seed(42)

# input_layer = Input(shape=(n_features, 1), name="Input")

# # -------- Block 1 --------
# b1_conv3 = Conv1D(32, 3, padding='same', activation='relu', name="B1_Conv3")(input_layer)
# b1_conv5 = Conv1D(32, 5, padding='same', activation='relu', name="B1_Conv5")(input_layer)
# b1 = Concatenate(axis=-1, name="B1_Concat")([b1_conv3, b1_conv5])
# b1 = BatchNormalization(name="B1_BN")(b1)
# b1 = Dropout(0.3, name="B1_Drop")(b1)

# # -------- Block 2 --------
# b2_conv3 = Conv1D(64, 3, padding='same', activation='relu', name="B2_Conv3")(b1)
# b2_conv5 = Conv1D(64, 5, padding='same', activation='relu', name="B2_Conv5")(b1)
# b2 = Concatenate(axis=-1, name="B2_Concat")([b2_conv3, b2_conv5])
# b2 = BatchNormalization(name="B2_BN")(b2)
# b2 = MaxPooling1D(pool_size=2, name="B2_Pool")(b2)
# b2 = Dropout(0.3, name="B2_Drop")(b2)

# # -------- Aggregation --------
# gap = GlobalAveragePooling1D(name="GAP")(b2)

# # -------- Classification Head --------
# head = Dense(128, activation='relu', name="Dense_128")(gap)
# head = Dropout(0.4, name="Head_Drop1")(head)
# head = Dense(64, activation='relu', name="Dense_64")(head)
# head = Dropout(0.3, name="Head_Drop2")(head)
# output = Dense(num_classes, activation='softmax', name="Output")(head)

# model = Model(inputs=input_layer, outputs=output, name="CNN_IDS_designed_34f")
# model.compile(
#     optimizer=Adam(learning_rate=1e-3),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )
# model.summary()


# # ============================================================================
# # CALLBACKS
# # ============================================================================
# from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# checkpoint = ModelCheckpoint(
#     "cnn_designed_34_best.h5",
#     monitor='val_accuracy',
#     save_best_only=True,
#     verbose=1
# )

# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=10,
#     restore_best_weights=True,
#     verbose=1
# )

# reduce_lr = ReduceLROnPlateau(
#     monitor='val_loss',
#     factor=0.5,
#     patience=5,
#     min_lr=1e-6,
#     verbose=1
# )


# # ============================================================================
# # TRAINING
# # ============================================================================
# import time

# start_train = time.time()

# history = model.fit(
#     X_train_cnn, y_train,
#     epochs=100,
#     batch_size=256,
#     validation_split=0.1,
#     class_weight=class_weights,
#     callbacks=[checkpoint, early_stop, reduce_lr],
#     verbose=1
# )

# train_time = time.time() - start_train
# print(f"\nTraining time: {train_time:.2f} s")


# # ============================================================================
# # COMPLETE TRAINING HISTORY
# # ============================================================================
# print("\n" + "=" * 78)
# print("COMPLETE TRAINING HISTORY — designed_34 (5 macro-classes)")
# print("=" * 78)
# print(f"{'Epoch':<8}{'Train Acc':<14}{'Val Acc':<14}{'Train Loss':<14}{'Val Loss':<14}")
# print("-" * 78)

# for epoch in range(len(history.history['accuracy'])):
#     print(
#         f"{epoch + 1:<8}"
#         f"{history.history['accuracy'][epoch]:<14.4f}"
#         f"{history.history['val_accuracy'][epoch]:<14.4f}"
#         f"{history.history['loss'][epoch]:<14.4f}"
#         f"{history.history['val_loss'][epoch]:<14.4f}"
#     )

# print("=" * 78)
# print(f"Total epochs run : {len(history.history['accuracy'])}")

# if len(history.history['val_accuracy']) > 0:
#     print(f"Best val accuracy: {max(history.history['val_accuracy']):.4f} "
#           f"(epoch {np.argmax(history.history['val_accuracy']) + 1})")
#     print(f"Best val loss    : {min(history.history['val_loss']):.4f} "
#           f"(epoch {np.argmin(history.history['val_loss']) + 1})")
# else:
#     print("No training history available.")

# print("=" * 78)


# # ============================================================================
# # EVALUATION
# # ============================================================================
# from sklearn.metrics import (
#     classification_report, confusion_matrix,
#     accuracy_score, precision_score, recall_score, f1_score
# )
# import seaborn as sns

# start_infer = time.time()
# y_pred_proba = model.predict(X_test_cnn, verbose=0)
# infer_time = time.time() - start_infer
# per_sample_ms = (infer_time / len(X_test_cnn)) * 1000

# y_pred = np.argmax(y_pred_proba, axis=1)

# acc  = accuracy_score(y_test, y_pred)
# prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
# rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
# f1w  = f1_score(y_test, y_pred, average='weighted', zero_division=0)
# f1m  = f1_score(y_test, y_pred, average='macro', zero_division=0)

# print("\n" + "="*60)
# print("TEST SET PERFORMANCE — designed_34 (7 PCs, 5 macro-classes)")
# print("="*60)
# print(f"Accuracy      : {acc:.4f}")
# print(f"Precision (W) : {prec:.4f}")
# print(f"Recall (W)    : {rec:.4f}")
# print(f"F1 (Weighted) : {f1w:.4f}")
# print(f"F1 (Macro)    : {f1m:.4f}")
# print(f"Training time : {train_time:.2f} s")
# print(f"Inference time: {infer_time:.2f} s ({per_sample_ms:.4f} ms/sample)")
# print(f"Model params  : {model.count_params()}")

# print("\n===== CLASSIFICATION REPORT =====")
# print(classification_report(y_test, y_pred, digits=4, zero_division=0))


# # ============================================================================
# # TRAINING AND VALIDATION CURVES
# # ============================================================================
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# axes[0].plot(history.history['accuracy'], label='Train accuracy')
# axes[0].plot(history.history['val_accuracy'], label='Validation accuracy')
# axes[0].set_title('Model Accuracy — designed_34 (5 macro-classes)')
# axes[0].set_xlabel('Epoch')
# axes[0].set_ylabel('Accuracy')
# axes[0].legend(loc='lower right')
# axes[0].grid(alpha=0.3)

# axes[1].plot(history.history['loss'], label='Train loss')
# axes[1].plot(history.history['val_loss'], label='Validation loss')
# axes[1].set_title('Model Loss — designed_34 (5 macro-classes)')
# axes[1].set_xlabel('Epoch')
# axes[1].set_ylabel('Loss')
# axes[1].legend(loc='upper right')
# axes[1].grid(alpha=0.3)

# plt.tight_layout()
# plt.savefig('training_curves_designed_34.png', dpi=300)
# plt.show()


# # ============================================================================
# # CONFUSION MATRIX
# # ============================================================================
# # Use the macro-class names in the order produced by macro_encoder
# class_names = list(macro_encoder.classes_)
# print("Confusion matrix labels:", class_names)

# cm = confusion_matrix(y_test, y_pred)

# plt.figure(figsize=(9, 7))
# sns.heatmap(
#     cm,
#     annot=True,
#     fmt='d',
#     cmap='Blues',
#     xticklabels=class_names,
#     yticklabels=class_names,
#     cbar_kws={'label': 'Count'}
# )
# plt.xlabel('Predicted Label')
# plt.ylabel('True Label')
# plt.title('Confusion Matrix — designed_34 (5 macro-classes)')
# plt.xticks(rotation=45, ha='right')
# plt.yticks(rotation=0)
# plt.tight_layout()
# plt.savefig('confusion_designed_34.png', dpi=300)
# plt.show()


# # ============================================================================
# # SAVE RESULTS
# # ============================================================================
# results = pd.DataFrame([{
#     'feature_set': 'designed_34_macro5',
#     'n_features': n_features,
#     'n_classes': num_classes,
#     'accuracy': acc,
#     'precision_weighted': prec,
#     'recall_weighted': rec,
#     'f1_weighted': f1w,
#     'f1_macro': f1m,
#     'train_time_s': train_time,
#     'inference_time_s': infer_time,
#     'inference_ms_per_sample': per_sample_ms,
#     'model_params': model.count_params()
# }])

# results.to_csv('cnn_results_designed_34_macro5.csv', index=False)
# print("\nSaved to cnn_results_designed_34_macro5.csv")
# results

DFtotal shape: (257673, 8)
Unique original classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Macro-class distribution:
attack_cat
DoS-Fuzzers        40599
Generic            58871
Normal             93000
System-Exploits    48365
Worm-Recon         16838
Name: count, dtype: int64

Macro-class integer mapping:
  0 -> DoS-Fuzzers
  1 -> Generic
  2 -> Normal
  3 -> System-Exploits
  4 -> Worm-Recon
Macro-class distribution:
attack_cat
0    40599
1    58871
2    93000
3    48365
4    16838
Name: count, dtype: int64

Train: (206138, 7) Test: (51535, 7)
x_train: (206138, 7) x_test: (51535, 7)
Class weights: {0: 1.269361741432926, 1: 0.8753763509353037, 2: 0.5541344086021506, 3: 1.0655329267031945, 4: 3.0606978470675577}
X_train_cnn: (206138, 7, 1)
X_test_cnn:  (51535, 7, 1)
num_classes: 5
n_features:  7
Artifacts directory ready: artifacts_designed_34/

Saved label encoder. Classes:
  0 -> DoS-Fuzzers
  1 -> Generic
  2 -> Normal
  3 -> System-Exploits
  4 -> Worm-Recon
Saved label_map.json

Saved f

In [3]:
import os

print("designed_34 artifacts:")
for f in sorted(os.listdir('artifacts_designed_34')):
    path = os.path.join('artifacts_designed_34', f)
    print(f"  {f}  ({os.path.getsize(path)} bytes)")

designed_34 artifacts:
  cnn_designed_34_best.h5  (797984 bytes)
  feature_order.json  (152 bytes)
  label_map.json  (106 bytes)
  macro_label_encoder.pkl  (530 bytes)
